[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SaronJenish/zawdwadw/blob/main/Experiment_26_Gibbs_Sampling_and_Contrastive_Divergence_Analysis.ipynb)


# Experiment 26: Gibbs Sampling and Contrastive Divergence Analysis

CSA47 – Deep Learning. Run the setup cell first, then execute the experiment cells in order.


In [ ]:
# Common setup – run this cell first
import os, time, warnings, math
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             confusion_matrix, mean_absolute_error, mean_squared_error)
np.random.seed(42)
plt.rcParams["figure.figsize"] = (7, 4)
print("Setup complete")

## Experiment 26 – Gibbs Sampling and Contrastive Divergence Analysis

In [ ]:
# Compare CD-k chains using the RBM functions and data from Experiment 25
def gibbs(v,k=1):
    hprob=sig(v@W+b); h=sample_prob(hprob)
    for _ in range(k):
        vprob=sig(h@W.T+a); v=sample_prob(vprob)
        hprob=sig(v@W+b); h=sample_prob(hprob)
    return v,h
for steps in [1,10,100,1000]:
    v=sample_prob(np.full((1,nv),.5))
    v,_=gibbs(v,steps)
    print(f"Visible sample after {steps} Gibbs steps:",v.astype(int))
# Short CD-k comparison; use a fresh parameter set for each k
rows=[]
for k in [1,5,10]:
    wk=rng.normal(0,.01,(nv,nh)); ak=np.zeros(nv); bk=np.zeros(nh); start=time.time()
    for ep in range(8):
        for st in range(0,len(V),30):
            v0=V[st:st+30]; hp=sig(v0@wk+bk); h=sample_prob(hp)
            for _ in range(k):
                vp=sig(h@wk.T+ak); v=sample_prob(vp); hp=sig(v@wk+bk); h=sample_prob(hp)
            wk+=.03*(v0.T@sig(v0@wk+bk)-v.T@hp)/len(v0)
            ak+=.03*np.mean(v0-v,axis=0); bk+=.03*np.mean(sig(v0@wk+bk)-hp,axis=0)
    reconstruction=sig(sig(V@wk+bk)@wk.T+ak)
    rows.append([f"CD-{k}",np.mean((V-reconstruction)**2),time.time()-start])
print(pd.DataFrame(rows,columns=["Method","Reconstruction error","Seconds"]))